# CliffForge — ChemProp 2.x Baseline on Dablander MMP Dataset

**Purpose**: Train ChemProp (absolute pIC50 D-MPNN) on the same 1,924-molecule Dablander MMP split,
evaluate with identical metrics, then merge with `summary_mmp.json` for a 3-model comparison.

**ChemProp version**: 2.x (numpy 2.x / torch 2.x / CUDA 13 compatible — no version pinning needed)

**Architecture**: Same directed message-passing neural network (D-MPNN) as ChemProp 1.x,
same default hyperparameters (depth=3, hidden=300, 60 epochs) — results are directly comparable.

> ⚡ Enable T4 GPU first: *Runtime → Change runtime type → T4 GPU*  
> 🕐 ~20 min total (install ~3 min, training ~10 min on GPU)

## 0 · Environment Check

In [ ]:
import subprocess, sys, torch
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU :', r.stdout.strip() if r.returncode==0 else 'None — training will be slow')
print('Python:', sys.version.split()[0])
print(f'torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')


## 1 · Install Dependencies

> ✅ ChemProp **2.x** is used here instead of 1.6.1 because:
> - ChemProp 1.x uses `numpy.VisibleDeprecationWarning` (removed in numpy 2.0)
> - ChemProp 1.x's transitive deps upgrade numpy back to 2.x no matter what order you install
> - ChemProp 2.x uses the **same D-MPNN architecture** with modern deps — no version conflicts

> ⚡ ~3 min. No numpy downgrade or runtime restart needed.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# ChemProp 2.x — numpy 2.x / torch 2.x / CUDA 13 compatible
print('Installing chemprop 2.x ...')
pip('chemprop>=2.0.4')

# RDKit (for SMILES processing in preprocessing)
print('Installing rdkit ...')
pip('rdkit')

# scikit-learn + scipy for metrics
pip('scikit-learn', 'scipy')

print('\nDone.')


In [ ]:
import importlib
for pkg in ['numpy','scipy','pandas','sklearn','rdkit','torch','chemprop']:
    try:
        m = importlib.import_module(pkg)
        print(f'  OK  {pkg} {getattr(m,"__version__","?")}')
    except ImportError as e:
        print(f'  MISSING {pkg}: {e}')
import torch
print(f'\nCUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU : {torch.cuda.get_device_name(0)}')


## 2 · Clone CliffForge Repository

In [ ]:
import os, sys
REPO = '/content/TELEMEDICON26'
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/chidwipak/TELEMEDICON26.git {REPO}
else:
    print('Already cloned.')
src = os.path.join(REPO, 'src')
if src not in sys.path: sys.path.insert(0, src)
from cliffforge import metrics as M
print('cliffforge.metrics imported OK')


## 3 · Download & Pre-Process Data

**Identical to `CliffForge_MMP_Colab.ipynb`** — same SEED=17, same scaffold split, same cliff definition.

In [ ]:
import math, numpy as np, pandas as pd
from collections import defaultdict
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

os.makedirs('/content/mmp_data/raw', exist_ok=True)
BASE = 'https://raw.githubusercontent.com/MarkusFerdinandDablander/QSAR-activity-cliff-experiments/main/data/postera_sars_cov_2_mpro'
!wget -q -O /content/mmp_data/raw/molecule_data_clean.csv {BASE}/molecule_data_clean.csv
!wget -q -O /content/mmp_data/raw/MMP_data_clean.csv      {BASE}/MMP_data_clean.csv

mol_raw = pd.read_csv('/content/mmp_data/raw/molecule_data_clean.csv')
mmp_raw = pd.read_csv('/content/mmp_data/raw/MMP_data_clean.csv')
print(f'Molecules: {len(mol_raw):,}  MMP pairs: {len(mmp_raw):,}')


In [ ]:
def canon2d(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return None, None
    flat = Chem.MolToSmiles(m, isomericSmiles=False)
    try: scaf = Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(flat)))
    except: scaf = 'NONE'
    return flat, scaf

records = []
for _, row in mol_raw.iterrows():
    ic50 = float(row['f_avg_IC50 [uM]'])
    if ic50 <= 0: continue
    flat, scaf = canon2d(row['SMILES'])
    if flat is None: continue
    records.append({'smiles': flat, 'pIC50': 6.0 - math.log10(ic50), 'scaf': scaf})

by_smi = defaultdict(list)
for r in records: by_smi[r['smiles']].append(r['pIC50'])
deduped = [{'smiles': s, 'pIC50': float(np.mean(v)),
             'scaf': next(r['scaf'] for r in records if r['smiles']==s)}
           for s,v in by_smi.items()]

# Scaffold split — SEED=17, identical to MMP notebook
MIN_SERIES, TEST_FRAC, SPLIT_SEED = 8, 0.35, 17
rng = np.random.RandomState(SPLIT_SEED)
scaf_members = defaultdict(list)
for rec in deduped: scaf_members[rec['scaf']].append(rec)

for scaf, members in scaf_members.items():
    sz = len(members)
    if sz >= MIN_SERIES:
        idx = rng.permutation(sz)
        n_test = min(max(3, int(round(sz*TEST_FRAC))), sz-3)
        test_ids = set(idx[:n_test].tolist())
        for i, rec in enumerate(members):
            rec['series'], rec['series_size'] = scaf, sz
            rec['split'] = 'test' if i in test_ids else 'train'
    else:
        for rec in members:
            rec['series'] = scaf if sz>1 else 'singleton'
            rec['series_size'] = sz
            rec['split'] = 'train'

df_all = pd.DataFrame(deduped)
n_train = int((df_all.split=='train').sum())
n_test  = int((df_all.split=='test').sum())
print(f'Total: {len(df_all):,}  Train: {n_train:,}  Test: {n_test:,}')
assert n_train == 1582 and n_test == 181, (
    f'Split mismatch (got {n_train}/{n_test}). Must match MMP notebook (1582/181).')
print('Split matches MMP notebook. OK.')


In [ ]:
smi_to_split  = dict(zip(df_all.smiles, df_all.split))
smi_to_pic50  = dict(zip(df_all.smiles, df_all.pIC50))
smi_to_series = dict(zip(df_all.smiles, df_all.series))

def canon_flat(smi):
    m = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(m, isomericSmiles=False) if m else None

CLIFF_SIM, CLIFF_DELTA = 0.80, 1.0
mmp_rows, skipped = [], 0
for _, row in mmp_raw.iterrows():
    s1, s2 = canon_flat(row['smiles_1']), canon_flat(row['smiles_2'])
    if not s1 or not s2 or s1==s2: skipped+=1; continue
    if s1 not in smi_to_split or s2 not in smi_to_split: skipped+=1; continue
    sp1, sp2 = smi_to_split[s1], smi_to_split[s2]
    pt = ('train_train' if sp1==sp2=='train' else 'test_test' if sp1==sp2=='test' else 'test_train')
    mmp_rows.append({'smiles_1':s1,'smiles_2':s2,
                     'pIC50_1':smi_to_pic50[s1],'pIC50_2':smi_to_pic50[s2],
                     'delta_pIC50':smi_to_pic50[s1]-smi_to_pic50[s2],
                     'abs_delta':abs(smi_to_pic50[s1]-smi_to_pic50[s2]),
                     'pair_type':pt,
                     'series_1':smi_to_series[s1],'series_2':smi_to_series[s2],
                     'same_series':smi_to_series[s1]==smi_to_series[s2]})

df_mmp = pd.DataFrame(mmp_rows)

# Tanimoto + cliff flags
smi_list = list(df_all.smiles)
fps      = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s),2,2048) for s in smi_list]
smi_to_fp = dict(zip(smi_list, fps))
df_mmp['tanimoto'] = [DataStructs.TanimotoSimilarity(smi_to_fp[r.smiles_1],smi_to_fp[r.smiles_2])
                      for r in df_mmp.itertuples()]
df_mmp['is_cliff'] = (df_mmp.tanimoto>=CLIFF_SIM)&(df_mmp.abs_delta>=CLIFF_DELTA)

for pt in ['train_train','test_train','test_test']:
    s = df_mmp[df_mmp.pair_type==pt]
    print(f'  {pt}: {len(s):,} pairs  {s.is_cliff.sum():,} cliffs')


## 4 · Train ChemProp 2.x

Uses ChemProp 2.x's **Python API** (Lightning-based) — no CLI, no version conflicts.
Same D-MPNN architecture as ChemProp 1.x with matching default hyperparameters.

> ⏱ ~10 min on T4 GPU  |  ~30 min on CPU

In [ ]:
import torch
from chemprop.data import MoleculeDatapoint, MoleculeDataset, build_dataloader

DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

df_train = df_all[df_all.split=='train'].reset_index(drop=True)
df_test  = df_all[df_all.split=='test'].reset_index(drop=True)

def make_dataset(df, has_targets=True):
    pts = []
    for _, row in df.iterrows():
        y = [[float(row['pIC50'])]] if has_targets else None
        try:
            pts.append(MoleculeDatapoint.from_smi(row['smiles'], y))
        except Exception:
            pts.append(None)
    valid = [p for p in pts if p is not None]
    print(f'  Dataset: {len(valid):,} valid / {len(pts):,} total')
    return MoleculeDataset(valid)

print('Building train dataset...')
train_dset = make_dataset(df_train, has_targets=True)
train_loader = build_dataloader(train_dset, shuffle=True, num_workers=0)

print('Building val dataset (test set, for monitoring only)...')
val_dset   = make_dataset(df_test,  has_targets=True)
val_loader = build_dataloader(val_dset,   shuffle=False, num_workers=0)

print('Datasets ready.')


In [ ]:
import lightning.pytorch as pl
from chemprop.models import MPNN
from chemprop.nn import BondMessagePassing, MeanAggregation, RegressionFFN

# Same hyperparameters as original CliffForge ChemProp baseline (depth=3, hidden=300)
mp  = BondMessagePassing(d_h=300, depth=3, dropout=0.0)
agg = MeanAggregation()
ffn = RegressionFFN(input_dim=300, hidden_dim=300, n_layers=2, dropout=0.0)
model = MPNN(message_passing=mp, agg=agg, predictor=ffn)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

trainer = pl.Trainer(
    max_epochs=60,
    accelerator=DEVICE,
    devices=1,
    enable_progress_bar=True,
    enable_model_summary=False,
    log_every_n_steps=10,
    default_root_dir='/content/chemprop_logs',
)

print(f'Training for 60 epochs on {DEVICE}...')
trainer.fit(model, train_loader, val_loader)
print('Training complete.')


## 5 · Predict pIC50 for All Molecules

In [ ]:
print('Predicting for all molecules...')

all_dset   = make_dataset(df_all, has_targets=False)
all_loader = build_dataloader(all_dset, shuffle=False, num_workers=0)

raw_preds = trainer.predict(model, all_loader)
# Each element is a Tensor of shape (batch, 1)
cp_pred_all = torch.cat(raw_preds).numpy().flatten()

print(f'Predictions: {len(cp_pred_all):,}')

# Coverage check
if len(cp_pred_all) != len(df_all):
    print(f'WARNING: {len(df_all)-len(cp_pred_all)} molecules had no prediction (invalid SMILES).')
    # Pad with NaN for missing molecules
    cp_full = np.full(len(df_all), np.nan)
    valid_idx = [i for i, r in enumerate(df_all.itertuples())
                 if Chem.MolFromSmiles(r.smiles) is not None]
    for j, i in enumerate(valid_idx):
        if j < len(cp_pred_all): cp_full[i] = cp_pred_all[j]
    cp_pred_all = cp_full

smi_to_pred_cp = dict(zip(df_all.smiles, cp_pred_all))

# Quick RMSE on test set
test_mask = (df_all.split=='test').values
y_test = df_all.pIC50.values[test_mask]
p_test = cp_pred_all[test_mask]
ok     = np.isfinite(p_test)
rmse   = float(np.sqrt(np.mean((y_test[ok]-p_test[ok])**2)))
corr   = float(np.corrcoef(y_test[ok], p_test[ok])[0,1])
print(f'\nTest RMSE: {rmse:.4f} pIC50  |  Pearson r: {corr:.4f}')


In [ ]:
# Add ChemProp pair-delta predictions to MMP test pairs
df_eval = df_mmp[df_mmp.pair_type!='train_train'].copy().reset_index(drop=True)

df_eval['pred_cp_1'] = [smi_to_pred_cp.get(s, np.nan) for s in df_eval.smiles_1]
df_eval['pred_cp_2'] = [smi_to_pred_cp.get(s, np.nan) for s in df_eval.smiles_2]
df_eval['pred_chemprop_delta'] = df_eval['pred_cp_1'] - df_eval['pred_cp_2']

valid = df_eval['pred_chemprop_delta'].notna().sum()
print(f'Valid pair predictions: {valid:,} / {len(df_eval):,}')

os.makedirs('/content/results_mmp', exist_ok=True)
df_eval.to_csv('/content/results_mmp/chemprop_test_predictions.csv', index=False, float_format='%.4f')
print('Saved → /content/results_mmp/chemprop_test_predictions.csv')


## 6 · Evaluate ChemProp

Using `cliffforge.metrics` — identical functions to the MMP notebook for fair comparison.

In [ ]:
dir_results_cp = {}
print(f'  {"Subset":<40} {"n":>6} {"DirAcc":>8} {"p":>10}')
print('  ' + '─'*68)
for pt in ['test_test','test_train']:
    sub = df_eval[df_eval.pair_type==pt].dropna(subset=['pred_chemprop_delta'])
    for min_d, lbl in [(0.0,'all'),(0.4,'|d|>=0.4'),(1.0,'cliff(|d|>=1.0)')]:
        s2 = sub[sub.abs_delta>=min_d]
        if len(s2)<3: continue
        key = f'{pt}|{lbl}'
        r = M.direction_accuracy(s2.delta_pIC50.values, s2.pred_chemprop_delta.values)
        dir_results_cp[key] = r
        if r['direction_accuracy'] is not None:
            p = r['binom_p_vs_0.5']
            sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
            print(f'  {key:<40} {r["n"]:>6} {r["direction_accuracy"]:>8.3f} {p:>10.4f} {sig}')


In [ ]:
cliff_results_cp = {}
TAU = 1.0
print(f'Cliff detection (tan>={CLIFF_SIM}, |Δ|>={CLIFF_DELTA}, tau={TAU})\n')
for pt in ['test_test','test_train']:
    sub = df_eval[(df_eval.pair_type==pt)&(df_eval.tanimoto>=CLIFF_SIM)].dropna(subset=['pred_chemprop_delta'])
    if len(sub)==0: continue
    n_cl = sub.is_cliff.sum()
    print(f'── {pt}  close={len(sub):,}  cliffs={n_cl}')
    m = M.cliff_detection_metrics(sub.is_cliff.values, np.abs(sub.pred_chemprop_delta.values), TAU)
    cliff_results_cp[pt] = m
    ap_s = f"{m.get('ap',float('nan')):.3f}" if 'ap' in m else 'N/A'
    print(f'  chemprop_abs:  AP={ap_s}  F1={m["f1"]:.3f}  '
          f'P={m["precision"]:.3f}  R={m["recall"]:.3f}  tp={m["tp"]} fn={m["fn"]}')


In [ ]:
from scipy.stats import wilcoxon

test_mask_bool = (df_all.split=='test').values
y_all     = df_all.pIC50.values.astype(float)
series_a  = df_all.series.values

ws_cp = M.within_series_spearman(
    true_y     = y_all[test_mask_bool],
    pred_y     = cp_pred_all[test_mask_bool],
    series_ids = series_a[test_mask_bool],
    min_series_size=5,
)
print(f'Within-series Spearman ρ (ChemProp 2.x):')
print(f'  Series : {ws_cp["n_series"]}')
print(f'  Mean ρ : {ws_cp["mean_rho"]:.4f}')
print(f'  Per ser: {[", ".join(f"{x:.2f}" for x in ws_cp["per_series"])]}')


## 7 · Combine with Existing MMP Results

Upload `summary_mmp.json` from `CliffForge_MMP_Colab.ipynb` (in your Downloads folder).

In [ ]:
from google.colab import files as colab_files
import json, copy
print('Upload summary_mmp.json from the first notebook:')
uploaded = colab_files.upload()
fname = list(uploaded.keys())[0]
base_summary = json.loads(uploaded[fname])
print(f'Loaded: {fname}  keys: {list(base_summary.keys())}')


In [ ]:
def safe_json(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(f'{type(o)}')

combined = copy.deepcopy(base_summary)

# ── Cliff detection ────────────────────────────────────────────────────────
combined.setdefault('cliff_detection',{}).update({
    'test_test|chemprop_abs' : cliff_results_cp.get('test_test', {}),
    'test_train|chemprop_abs': cliff_results_cp.get('test_train', {}),
})

# ── Direction accuracy ─────────────────────────────────────────────────────
for subset, res in dir_results_cp.items():
    combined.setdefault('direction_accuracy',{}).setdefault(subset,{})['chemprop_abs'] = res

# ── Within-series Spearman ─────────────────────────────────────────────────
ws = combined.setdefault('within_series_spearman', {})
ws['chemprop_abs_mean']     = ws_cp['mean_rho']
ws['chemprop_abs_per_series'] = ws_cp['per_series']
ws['chemprop_abs_n_series']   = ws_cp['n_series']

# ── ChemProp meta ──────────────────────────────────────────────────────────
combined['chemprop_abs_meta'] = {
    'version': '2.x', 'architecture': 'D-MPNN depth=3 hidden=300',
    'epochs': 60, 'n_train': n_train,
    'test_rmse': rmse, 'test_pearson_r': corr,
}

out_path = '/content/results_mmp/summary_mmp_combined.json'
with open(out_path,'w') as f:
    json.dump(combined, f, indent=2, default=safe_json)
print(f'Saved → {out_path}  ({os.path.getsize(out_path):,} bytes)')


## 8 · Full Three-Model Comparison Table

In [ ]:
def g_ap(d, k):  return d.get('cliff_detection',{}).get(k,{}).get('ap', float('nan'))
def g_f1(d, k):  return d.get('cliff_detection',{}).get(k,{}).get('f1', float('nan'))
def g_da(d, s, m): return d.get('direction_accuracy',{}).get(s,{}).get(m,{}).get('direction_accuracy', float('nan'))

ws = combined.get('within_series_spearman',{})

print('='*90)
print('  THREE-MODEL COMPARISON — Dablander MMP Dataset (1,924 molecules)')
print('='*90)
print(f'{"Metric":<50} {"CliffForge":>12} {"ChemProp":>12} {"5-NN":>10}')
print('─'*90)

rows = [
    ('Within-series ρ (test, ≥5 members)',
     ws.get('rel_mean',float('nan')),  ws.get('chemprop_abs_mean',float('nan')), ws.get('knn_mean',float('nan'))),
    ('Direction acc — test-test, all pairs',
     g_da(combined,'test_test|all','rel_siamese'), g_da(combined,'test_test|all','chemprop_abs'), g_da(combined,'test_test|all','knn_5nn')),
    ('Direction acc — test-test, cliff (|Δ|≥1.0)',
     g_da(combined,'test_test|cliff(|d|>=1.0)','rel_siamese'), g_da(combined,'test_test|cliff(|d|>=1.0)','chemprop_abs'), g_da(combined,'test_test|cliff(|d|>=1.0)','knn_5nn')),
    ('AP — test-test / strict (2 cliffs)',
     g_ap(combined,'test_test|rel_siamese'), g_ap(combined,'test_test|chemprop_abs'), g_ap(combined,'test_test|knn_5nn')),
    ('F1 — test-test / strict',
     g_f1(combined,'test_test|rel_siamese'), g_f1(combined,'test_test|chemprop_abs'), g_f1(combined,'test_test|knn_5nn')),
    ('AP — test-train / lead-opt (25 cliffs)',
     g_ap(combined,'test_train|rel_siamese'), g_ap(combined,'test_train|chemprop_abs'), g_ap(combined,'test_train|knn_5nn')),
    ('F1 — test-train / lead-opt',
     g_f1(combined,'test_train|rel_siamese'), g_f1(combined,'test_train|chemprop_abs'), g_f1(combined,'test_train|knn_5nn')),
]

for name, cf, cp, knn in rows:
    vals = [v for v in [cf, cp, knn] if np.isfinite(v)]
    best = max(vals) if vals else None
    def fmt(v):
        s = f'{v:.3f}' if np.isfinite(v) else 'N/A'
        return f'{s} ★' if (np.isfinite(v) and v==best) else f'{s}'
    print(f'{name:<50} {fmt(cf):>12} {fmt(cp):>12} {str("N/A" if not np.isfinite(knn) else f"{knn:.3f}"):>10}')

print('─'*90)
print(f'ChemProp test RMSE : {rmse:.4f} pIC50  |  Pearson r: {corr:.4f}')
print(f'ChemProp model ver.: 2.x (D-MPNN depth=3 hidden=300 60 epochs)')
print('★ = best in row')


## 9 · Download Results

In [ ]:
!zip -j /content/cliffforge_combined_results.zip \
    /content/results_mmp/summary_mmp_combined.json \
    /content/results_mmp/chemprop_test_predictions.csv
from google.colab import files as colab_files
colab_files.download('/content/cliffforge_combined_results.zip')
print('Download triggered.')


---
## Done ✓

**Outputs:**
- `summary_mmp_combined.json` — CliffForge + ChemProp + 5-NN all in one file
- `chemprop_test_predictions.csv` — per-pair delta predictions from ChemProp

**Context for interpreting results:**
- Original prototype: ChemProp ρ=0.192 vs CliffForge ρ=0.581 (within-series ranking)
- Original prototype: ChemProp lead-opt AP=0.570 vs CliffForge AP=0.466
- Now the question is: does CliffForge's larger dataset advantage persist vs ChemProp 2.x?